In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

# ============================================================
# NOTEBOOK 09 — PRIMARY STATISTICAL ANALYSIS
# ONE CELL / ONE RUN
#
# Purpose:
#   - Do NOT retrain models.
#   - Load the frozen models produced earlier in the pipeline.
#   - Recreate the severe-condition paired outcomes.
#   - Run the primary 5-seed hierarchical paired bootstrap.
#   - Save the machine-readable 09_* statistical outputs.
#
# Primary severe condition:
#   50% detected-AP loss + sigma = 10 dB
#
# Primary corruption seeds:
#   42, 43, 44, 45, 46
#
# Bootstrap:
#   5,000 replicates
# ============================================================


# ------------------------------------------------------------
# 1. Portable repository root
# ------------------------------------------------------------

def find_project_root():
    start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "models").is_dir()
            and (candidate / "results").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate repository root. "
        "Run this notebook from inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_CSV = PROJECT_ROOT / "results" / "csv"

TRAIN_FILE = DATA_RAW / "trainingData.csv"
TEST_FILE = DATA_RAW / "validationData.csv"

RESULTS_CSV.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT:", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Constants
# ------------------------------------------------------------

MISSING_ORIGINAL = 100
MISSING_RSSI = -110.0

MIN_OBSERVED_RSSI = -104.0
MAX_OBSERVED_RSSI = 0.0

AP_LOSS = 0.50
RSSI_SIGMA = 10.0

EVAL_SEEDS = [42, 43, 44, 45, 46]

N_BOOT = 5000
BOOTSTRAP_SEED = 42


# ------------------------------------------------------------
# 3. Find frozen primary model/scaler artifacts
# ------------------------------------------------------------

def load_joblib_candidates():
    items = []

    for path in sorted(MODEL_DIR.rglob("*.joblib")):
        text = str(path).lower()

        # Reviewer-control models are NOT part of the original primary run.
        if (
            "11_equal_volume" in text
            or "12_random_forest" in text
            or "review" in text
        ):
            continue

        try:
            obj = joblib.load(path)
        except Exception:
            continue

        items.append((path, obj))

    return items


artifacts = load_joblib_candidates()


def choose_exact_filename(filename, expected_type):
    matches = [
        (p, o)
        for p, o in artifacts
        if p.name == filename
        and isinstance(o, expected_type)
    ]

    if not matches:
        return None, None

    # Prefer the shallowest/earliest primary artifact if duplicates exist.
    matches = sorted(
        matches,
        key=lambda x: (
            len(x[0].parts),
            str(x[0])
        )
    )

    return matches[0]


def choose_by_tokens(expected_type, required_tokens, preferred_tokens=()):
    candidates = []

    for path, obj in artifacts:
        if not isinstance(obj, expected_type):
            continue

        text = str(path).lower()

        if not all(tok.lower() in text for tok in required_tokens):
            continue

        score = 0

        for tok in preferred_tokens:
            if tok.lower() in text:
                score += 1

        candidates.append(
            (score, path, obj)
        )

    if not candidates:
        return None, None

    candidates.sort(
        key=lambda x: (
            -x[0],
            len(x[1].parts),
            str(x[1])
        )
    )

    _, path, obj = candidates[0]

    return path, obj


# Baseline filenames are documented in the earlier project code.
clean_clf_path, clean_classifier = choose_exact_filename(
    "mlp_classifier.joblib",
    MLPClassifier
)

clean_reg_path, clean_regressor = choose_exact_filename(
    "mlp_regressor.joblib",
    MLPRegressor
)

coord_scaler_path, coordinate_scaler = choose_exact_filename(
    "coordinate_scaler.joblib",
    StandardScaler
)

# Feature scaler name varied across revisions; find the primary MinMaxScaler.
feature_scaler_candidates = [
    (p, o)
    for p, o in artifacts
    if isinstance(o, MinMaxScaler)
]

if feature_scaler_candidates:
    feature_scaler_candidates.sort(
        key=lambda x: (
            "feature" not in x[0].name.lower(),
            len(x[0].parts),
            str(x[0])
        )
    )

    feature_scaler_path, feature_scaler = feature_scaler_candidates[0]
else:
    feature_scaler_path = None
    feature_scaler = None


# Robust classifier: AP-dropout MLP classifier.
robust_clf_path, robust_classifier = choose_by_tokens(
    MLPClassifier,
    required_tokens=("drop",),
    preferred_tokens=("ap", "classifier", "robust", "final")
)

# Robust regressor: combined-augmentation MLP regressor.
robust_reg_path, robust_regressor = choose_by_tokens(
    MLPRegressor,
    required_tokens=("combined",),
    preferred_tokens=("regressor", "robust", "final")
)


required = {
    "clean_classifier": (clean_clf_path, clean_classifier),
    "robust_classifier": (robust_clf_path, robust_classifier),
    "clean_regressor": (clean_reg_path, clean_regressor),
    "robust_regressor": (robust_reg_path, robust_regressor),
}

missing = [
    name
    for name, (path, obj) in required.items()
    if path is None or obj is None
]

if missing:
    print("\nAvailable primary model artifacts:")
    for path, obj in artifacts:
        print(" -", path.relative_to(PROJECT_ROOT), type(obj).__name__)

    raise FileNotFoundError(
        "Could not identify: " + ", ".join(missing)
    )


print("\nFROZEN PRIMARY ARTIFACTS")
for name, (path, _) in required.items():
    print(
        f"{name:20s}:",
        path.relative_to(PROJECT_ROOT)
    )

if feature_scaler_path is not None:
    print(
        f"{'feature_scaler':20s}:",
        feature_scaler_path.relative_to(PROJECT_ROOT)
    )
else:
    print(
        f"{'feature_scaler':20s}:",
        "not saved — will reconstruct from original training split"
    )

if coord_scaler_path is not None:
    print(
        f"{'coordinate_scaler':20s}:",
        coord_scaler_path.relative_to(PROJECT_ROOT)
    )
else:
    print(
        f"{'coordinate_scaler':20s}:",
        "not saved — will reconstruct from original training split"
    )


# ------------------------------------------------------------
# 4. Recreate the original preprocessing split and load test data
#
# The historical primary pipeline did not necessarily save the
# MinMax feature scaler as a standalone .joblib artifact.
# When it is absent, reconstruct it exactly from the original
# 15,949-sample training split:
#   - 80/20 split
#   - random_state = 42
#   - stratified by joint building-floor class
#
# This reproduces preprocessing only. It does NOT retrain models.
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

wifi_cols = [
    c for c in train_df.columns
    if c.upper().startswith("WAP")
]

assert len(train_df) == 19937
assert len(test_df) == 1111
assert len(wifi_cols) == 520


def make_class_labels(df):
    return (
        df["BUILDINGID"].astype(int).astype(str)
        + "_"
        + df["FLOOR"].astype(int).astype(str)
    ).to_numpy()


# ----- original training collection -----

X_train_original = train_df[
    wifi_cols
].to_numpy(dtype=float)

train_observed_mask = (
    X_train_original
    != MISSING_ORIGINAL
)

X_train_raw = X_train_original.copy()

X_train_raw[
    ~train_observed_mask
] = MISSING_RSSI

y_train_class = make_class_labels(
    train_df
)

y_train_coord = train_df[
    ["LONGITUDE", "LATITUDE"]
].to_numpy(dtype=float)


all_indices = np.arange(
    len(train_df)
)

train_indices, dev_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=y_train_class
)

assert len(train_indices) == 15949
assert len(dev_indices) == 3988


# Reconstruct feature scaler only when no saved primary scaler exists.
if feature_scaler is None:
    feature_scaler = MinMaxScaler()

    feature_scaler.fit(
        X_train_raw[
            train_indices
        ]
    )

    print(
        "\nReconstructed primary MinMaxScaler "
        "from the original 15,949-sample training split."
    )


# Reconstruct coordinate scaler only when no saved primary scaler exists.
if coordinate_scaler is None:
    coordinate_scaler = StandardScaler()

    coordinate_scaler.fit(
        y_train_coord[
            train_indices
        ]
    )

    print(
        "Reconstructed primary coordinate StandardScaler "
        "from the original 15,949-sample training split."
    )


# ----- held-out final test collection -----

X_original = test_df[
    wifi_cols
].to_numpy(dtype=float)

observed_mask = (
    X_original
    != MISSING_ORIGINAL
)

X_raw = X_original.copy()

X_raw[
    ~observed_mask
] = MISSING_RSSI

y_class = make_class_labels(
    test_df
)

y_coord = test_df[
    ["LONGITUDE", "LATITUDE"]
].to_numpy(dtype=float)


# ------------------------------------------------------------
# 5. Corruption functions
# ------------------------------------------------------------

def apply_ap_loss(X, mask, rng, fraction):
    X_out = X.copy()
    surviving = mask.copy()

    for i in range(X.shape[0]):
        observed = np.flatnonzero(mask[i])

        if len(observed) == 0:
            continue

        n_drop = int(
            np.floor(
                fraction * len(observed)
            )
        )

        if fraction > 0:
            n_drop = max(1, n_drop)

        n_drop = min(
            n_drop,
            len(observed)
        )

        if n_drop == 0:
            continue

        drop_idx = rng.choice(
            observed,
            size=n_drop,
            replace=False
        )

        X_out[i, drop_idx] = MISSING_RSSI
        surviving[i, drop_idx] = False

    return X_out, surviving


def apply_rssi_noise(X, surviving, rng, sigma):
    X_out = X.copy()

    for i in range(X.shape[0]):
        observed = np.flatnonzero(
            surviving[i]
        )

        if len(observed) == 0:
            continue

        noise = rng.normal(
            0.0,
            sigma,
            size=len(observed)
        )

        X_out[i, observed] += noise

        X_out[i, observed] = np.clip(
            X_out[i, observed],
            MIN_OBSERVED_RSSI,
            MAX_OBSERVED_RSSI
        )

    X_out[~surviving] = MISSING_RSSI

    return X_out


def make_severe_condition(seed):
    """
    Reproduce Notebook 08 exactly (the frozen primary evaluation):
      - AP loss uses np.random.default_rng(seed)
      - RSSI noise uses a SEPARATE np.random.default_rng(seed + 10000),
        drawn as one full-matrix normal sample and applied to surviving APs.
    """
    X_drop, _ = apply_ap_loss(
        X_raw,
        observed_mask,
        np.random.default_rng(seed),
        AP_LOSS
    )

    X_noisy = X_drop.copy().astype(float)
    surviving = observed_mask & (X_drop != MISSING_RSSI)

    noise = np.random.default_rng(seed + 10000).normal(
        loc=0.0,
        scale=RSSI_SIGMA,
        size=X_noisy.shape
    )

    X_noisy[surviving] += noise[surviving]
    X_noisy[surviving] = np.clip(
        X_noisy[surviving],
        MIN_OBSERVED_RSSI,
        MAX_OBSERVED_RSSI
    )
    X_noisy[~surviving] = MISSING_RSSI

    return X_noisy

def inverse_coordinate_prediction(model, X_scaled):
    pred_scaled = model.predict(X_scaled)

    return coordinate_scaler.inverse_transform(
        pred_scaled
    )


def euclidean_errors(y_true, y_pred):
    return np.sqrt(
        np.sum(
            (y_pred - y_true) ** 2,
            axis=1
        )
    )


# ------------------------------------------------------------
# 6. Clean paired comparison
# ------------------------------------------------------------

X_clean = feature_scaler.transform(X_raw)

clean_clf_pred = clean_classifier.predict(X_clean)
robust_clf_pred = robust_classifier.predict(X_clean)

clean_reg_pred = inverse_coordinate_prediction(
    clean_regressor,
    X_clean
)

robust_reg_pred = inverse_coordinate_prediction(
    robust_regressor,
    X_clean
)

clean_cls_effect = (
    (robust_clf_pred == y_class).astype(float)
    -
    (clean_clf_pred == y_class).astype(float)
)

clean_reg_error = euclidean_errors(
    y_coord,
    clean_reg_pred
)

robust_reg_error = euclidean_errors(
    y_coord,
    robust_reg_pred
)

clean_loc_effect = (
    clean_reg_error
    -
    robust_reg_error
)

print("\nCLEAN CONDITION")
print(
    "Classification:",
    round(accuracy_score(y_class, clean_clf_pred), 4),
    "->",
    round(accuracy_score(y_class, robust_clf_pred), 4),
    "effect:",
    round(clean_cls_effect.mean(), 4)
)

print(
    "Localization:",
    round(clean_reg_error.mean(), 4),
    "->",
    round(robust_reg_error.mean(), 4),
    "reduction:",
    round(clean_loc_effect.mean(), 4)
)


# ------------------------------------------------------------
# 7. Severe-condition paired outcomes over five seeds
# ------------------------------------------------------------

classification_effects = []
localization_effects = []

paired_rows = []
classification_seed_rows = []
localization_seed_rows = []

for seed in EVAL_SEEDS:
    X_severe_raw = make_severe_condition(seed)

    X_severe = feature_scaler.transform(
        X_severe_raw
    )

    base_cls_pred = clean_classifier.predict(
        X_severe
    )

    robust_cls_pred = robust_classifier.predict(
        X_severe
    )

    base_reg_pred = inverse_coordinate_prediction(
        clean_regressor,
        X_severe
    )

    robust_reg_pred = inverse_coordinate_prediction(
        robust_regressor,
        X_severe
    )

    base_correct = (
        base_cls_pred == y_class
    ).astype(float)

    robust_correct = (
        robust_cls_pred == y_class
    ).astype(float)

    cls_effect = (
        robust_correct
        -
        base_correct
    )

    base_error = euclidean_errors(
        y_coord,
        base_reg_pred
    )

    robust_error = euclidean_errors(
        y_coord,
        robust_reg_pred
    )

    loc_effect = (
        base_error
        -
        robust_error
    )

    classification_effects.append(
        cls_effect
    )

    localization_effects.append(
        loc_effect
    )

    classification_seed_rows.append({
        "Seed": seed,
        "Baseline_Accuracy": accuracy_score(
            y_class,
            base_cls_pred
        ),
        "Robust_Accuracy": accuracy_score(
            y_class,
            robust_cls_pred
        ),
        "Accuracy_Gain": cls_effect.mean(),
        "Baseline_Macro_F1": f1_score(
            y_class,
            base_cls_pred,
            average="macro",
            zero_division=0
        ),
        "Robust_Macro_F1": f1_score(
            y_class,
            robust_cls_pred,
            average="macro",
            zero_division=0
        ),
    })

    localization_seed_rows.append({
        "Seed": seed,
        "Baseline_Mean_Error": base_error.mean(),
        "Robust_Mean_Error": robust_error.mean(),
        "Error_Reduction": loc_effect.mean(),
    })

    for i in range(len(y_class)):
        paired_rows.append({
            "Seed": seed,
            "Fingerprint_Index": i,
            "Baseline_Classifier_Correct": base_correct[i],
            "Robust_Classifier_Correct": robust_correct[i],
            "Classification_Effect": cls_effect[i],
            "Baseline_Localization_Error": base_error[i],
            "Robust_Localization_Error": robust_error[i],
            "Localization_Effect": loc_effect[i],
        })


classification_effects = np.asarray(
    classification_effects
)

localization_effects = np.asarray(
    localization_effects
)

paired_df = pd.DataFrame(
    paired_rows
)

classification_by_seed_df = pd.DataFrame(
    classification_seed_rows
)

localization_by_seed_df = pd.DataFrame(
    localization_seed_rows
)


# ------------------------------------------------------------
# 8. Bootstrap functions
# ------------------------------------------------------------

def paired_bootstrap(
    effects,
    n_boot=5000,
    seed=42
):
    effects = np.asarray(
        effects,
        dtype=float
    )

    rng = np.random.default_rng(seed)

    n = len(effects)

    boot = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n
        )

        boot[b] = effects[idx].mean()

    estimate = effects.mean()

    low, high = np.percentile(
        boot,
        [2.5, 97.5]
    )

    return (
        float(estimate),
        float(low),
        float(high)
    )


def hierarchical_bootstrap(
    effect_matrix,
    n_boot=5000,
    seed=42
):
    effect_matrix = np.asarray(
        effect_matrix,
        dtype=float
    )

    n_seeds, n_samples = (
        effect_matrix.shape
    )

    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)

    for b in range(n_boot):
        sampled_seeds = rng.integers(
            0,
            n_seeds,
            size=n_seeds
        )

        seed_means = []

        for seed_idx in sampled_seeds:
            sample_idx = rng.integers(
                0,
                n_samples,
                size=n_samples
            )

            seed_means.append(
                effect_matrix[
                    seed_idx,
                    sample_idx
                ].mean()
            )

        boot[b] = np.mean(seed_means)

    estimate = effect_matrix.mean()

    low, high = np.percentile(
        boot,
        [2.5, 97.5]
    )

    return (
        float(estimate),
        float(low),
        float(high)
    )


# ------------------------------------------------------------
# 9. Run the four primary statistical comparisons
# ------------------------------------------------------------

clean_cls = paired_bootstrap(
    clean_cls_effect,
    N_BOOT,
    BOOTSTRAP_SEED
)

severe_cls = hierarchical_bootstrap(
    classification_effects,
    N_BOOT,
    BOOTSTRAP_SEED
)

clean_loc = paired_bootstrap(
    clean_loc_effect,
    N_BOOT,
    BOOTSTRAP_SEED
)

severe_loc = hierarchical_bootstrap(
    localization_effects,
    N_BOOT,
    BOOTSTRAP_SEED
)


summary_df = pd.DataFrame([
    {
        "Task": "Classification",
        "Condition": "Clean",
        "Estimate": clean_cls[0],
        "CI_Lower": clean_cls[1],
        "CI_Upper": clean_cls[2],
        "Bootstrap": "Paired"
    },
    {
        "Task": "Classification",
        "Condition": "50% AP loss + sigma=10 dB",
        "Estimate": severe_cls[0],
        "CI_Lower": severe_cls[1],
        "CI_Upper": severe_cls[2],
        "Bootstrap": "Hierarchical paired"
    },
    {
        "Task": "Localization",
        "Condition": "Clean",
        "Estimate": clean_loc[0],
        "CI_Lower": clean_loc[1],
        "CI_Upper": clean_loc[2],
        "Bootstrap": "Paired"
    },
    {
        "Task": "Localization",
        "Condition": "50% AP loss + sigma=10 dB",
        "Estimate": severe_loc[0],
        "CI_Lower": severe_loc[1],
        "CI_Upper": severe_loc[2],
        "Bootstrap": "Hierarchical paired"
    }
])


# ------------------------------------------------------------
# 10. Save Notebook 09 outputs
# ------------------------------------------------------------

paired_df.to_csv(
    RESULTS_CSV
    / "09_severe_paired_sample_results.csv",
    index=False
)

classification_by_seed_df.to_csv(
    RESULTS_CSV
    / "09_severe_classification_by_seed.csv",
    index=False
)

localization_by_seed_df.to_csv(
    RESULTS_CSV
    / "09_severe_localization_by_seed.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_CSV
    / "09_statistical_summary.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_CSV
    / "09_final_paper_statistical_results.csv",
    index=False
)


# ------------------------------------------------------------
# 11. Final output
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK 09 — PRIMARY STATISTICAL RESULTS")
print("=" * 80)

print(
    summary_df.round(6)
    .to_string(index=False)
)

print("\nExpected frozen manuscript point estimates:")
print("Clean classification:     -0.0054")
print("Severe classification:     0.0571")
print("Clean localization:        1.9402")
print("Severe localization:      14.9227")

print("\nExpected frozen manuscript CIs:")
print("Clean classification:    [-0.0180, 0.0063]")
print("Severe classification:   [ 0.0441, 0.0706]")
print("Clean localization:      [ 1.4704, 2.4083]")
print("Severe localization:     [14.2087,15.6291]")

print("\nSaved:")
print(" - 09_severe_paired_sample_results.csv")
print(" - 09_severe_classification_by_seed.csv")
print(" - 09_severe_localization_by_seed.csv")
print(" - 09_statistical_summary.csv")
print(" - 09_final_paper_statistical_results.csv")

print("\nNOTEBOOK 09 COMPLETE.")


PROJECT ROOT: /Users/am/Documents/ADU/wifi-localization-research

FROZEN PRIMARY ARTIFACTS
clean_classifier    : models/03_clean_baselines/mlp_classifier.joblib
robust_classifier   : models/07_robust_training/mlp_ap_dropout_classifier.joblib
clean_regressor     : models/03_clean_baselines/mlp_regressor.joblib
robust_regressor    : models/07_robust_training/mlp_combined_regressor.joblib
feature_scaler      : not saved — will reconstruct from original training split
coordinate_scaler   : models/03_clean_baselines/coordinate_scaler.joblib

Reconstructed primary MinMaxScaler from the original 15,949-sample training split.

CLEAN CONDITION
Classification: 0.9109 -> 0.9055 effect: -0.0054
Localization: 12.0538 -> 10.1136 reduction: 1.9402

NOTEBOOK 09 — PRIMARY STATISTICAL RESULTS
          Task                 Condition  Estimate  CI_Lower  CI_Upper           Bootstrap
Classification                     Clean -0.005401 -0.018002  0.006301              Paired
Classification 50% AP loss + sig